# Name That LLM: train the Mistral-7B-v0.1 LoRA member on Colab

Generated by `name_that_llm/remote_train.py colab`; do not edit by hand.

Trains the same 5-fold, two-stage LoRA member as the Kaggle training notebook, without using Kaggle GPU quota, then stages the output for upload as a private Kaggle dataset.

**Before running:**
1. **Runtime → Change runtime type → A100 GPU** (an L4 works, at about the speed of Kaggle's two T4s).
2. **Secrets** (key icon on the left): add `KAGGLE_USERNAME` and `KAGGLE_KEY` and give this notebook access. You must have joined the competition on Kaggle.
3. Run the timing run first (`MODE = 'timing'`, ~10 min), check its s/step, then set `MODE = 'full'` and run all again.

Keep this tab open while the full run trains: a disconnect stops it, and stage 1 has no checkpoint part-way, so it would start over.

In [ ]:
VARIANT = 'mistral7b-full'
MODE = 'timing'  # 'timing' (~10 min speed check) or 'full'
KAGGLE_USER = ''  # your Kaggle username, for the upload step
# the full run writes to Google Drive, so the result survives a disconnect
DRIVE_OUT = '/content/drive/MyDrive/ntl/lora_out_mistral7b_full'

## Setup

In [ ]:
import os
import subprocess

from google.colab import drive, userdata

print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'],
                     capture_output=True, text=True).stdout)
drive.mount('/content/drive')
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

In [ ]:
%cd /content
!test -d ntl || git clone https://github.com/kemalhanan/name-that-llm.git ntl
%cd /content/ntl
!git pull -q
!pip install -q uv
!uv sync -q --group lora

## Inputs

The competition data, the Open Pool and the base model, downloaded with kagglehub to this runtime's disk (about 15 GB for a 7-8B model).

In [ ]:
!uv run python -m name_that_llm.remote_train fetch {VARIANT}

## Train

Timing runs write to the runtime's disk and start fresh each time. The full run writes to `DRIVE_OUT` and refuses to start if that folder holds files, for example from a run that was cut off: delete it first.

In [ ]:
if MODE == 'full':
    OUT = DRIVE_OUT
else:
    OUT = '/content/timing_out'
    !rm -rf {OUT}
!uv run python -m name_that_llm.remote_train run {VARIANT} --mode {MODE} --out {OUT}

## Upload

Stages the full run's output with a `dataset-metadata.json`. The folder keeps the training notebook's name, which is how the submission notebook finds the member. Then upload it (the Kaggle CLI makes new datasets private) and attach it to the submission notebook.

In [ ]:
assert MODE == 'full', 'upload only a full run'
assert KAGGLE_USER, 'set KAGGLE_USER in the first cell'
!uv run python -m name_that_llm.remote_train package {VARIANT} --user {KAGGLE_USER} --out {DRIVE_OUT}
STAGE = '/content/ntl/data/remote/out/upload/lora_out_mistral7b_full'
!uv run --with kaggle kaggle datasets create -p {STAGE} --dir-mode tar